<a href="https://colab.research.google.com/github/2003Ankita/ObjectRecognizer/blob/main/part2_resnet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2: Fine-Tune a Pretrained CNN — OOD-Robust

Pretrained CNN fine-tuned on CIFAR-100 (224×224).  
Tries ConvNeXt-Base (IN-22k) → ResNet-101 → ResNet-50 (best available).  
**OOD improvements**: AugMix, ColorJitter, GaussianBlur, CutMix/MixUp, 7-view TTA.

In [ ]:
# Runtime configuration
FAST_DEV_RUN = False  # True = quick smoke test
SEED = 42

import os

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

WORK_ROOT = "/content" if IN_COLAB else os.path.abspath("./temp_student")
os.makedirs(WORK_ROOT, exist_ok=True)

DATA_ROOT = os.path.join(WORK_ROOT, "data")
OOD_DIR = os.path.join(WORK_ROOT, "ood-test-CS541")
SUBMISSION_PATH = os.path.join(WORK_ROOT, "submission_ood.csv")

print("IN_COLAB:", IN_COLAB)
print("WORK_ROOT:", WORK_ROOT)

In [ ]:
# Install required packages
import importlib.util, subprocess, sys
required = ["torch", "torchvision", "tqdm", "numpy", "pandas", "matplotlib", "huggingface_hub", "timm"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *missing])
print("Environment ready")

In [ ]:
import os, random, math, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm
import timm
import matplotlib.pyplot as plt
from PIL import ImageFilter, ImageOps

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

MEAN = (0.5071, 0.4867, 0.4408)  ### DO NOT CHANGE THIS
STD  = (0.2675, 0.2565, 0.2761)  ### DO NOT CHANGE THIS
USE_AMP = True

def _amp_ctx(device):
    enabled = USE_AMP and device.type == "cuda"
    try: return torch.amp.autocast(device_type=device.type, enabled=enabled)
    except (AttributeError, TypeError):
        from torch.cuda.amp import autocast; return autocast(enabled=enabled)

def _make_scaler(device):
    enabled = USE_AMP and device.type == "cuda"
    try: return torch.amp.GradScaler(device=device.type, enabled=enabled)
    except (AttributeError, TypeError):
        from torch.cuda.amp import GradScaler; return GradScaler(enabled=enabled)

# ============================================================
# Custom augmentation transforms for OOD robustness
# ============================================================
class GaussianBlurPIL:
    """Random Gaussian blur (simulates blur corruptions)."""
    def __init__(self, p=0.3, radius_range=(0.1, 2.0)):
        self.p = p; self.radius_range = radius_range
    def __call__(self, img):
        if random.random() < self.p:
            r = random.uniform(*self.radius_range)
            return img.filter(ImageFilter.GaussianBlur(radius=r))
        return img

class RandomSolarize:
    """Random solarize (simulates contrast corruptions)."""
    def __init__(self, p=0.1, threshold=128):
        self.p = p; self.threshold = threshold
    def __call__(self, img):
        if random.random() < self.p:
            return ImageOps.solarize(img, self.threshold)
        return img

class RandomPosterize:
    """Random posterize (simulates JPEG/quantization corruptions)."""
    def __init__(self, p=0.1, bits=4):
        self.p = p; self.bits = bits
    def __call__(self, img):
        if random.random() < self.p:
            return ImageOps.posterize(img, self.bits)
        return img

set_seed(SEED)
device = get_device()
print("Device:", device)
if device.type=="cuda":
    print("GPU:", torch.cuda.get_device_name(), "| VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")
print("PyTorch:", torch.__version__)

In [ ]:
# ============================================================
# Data loaders with STRONG OOD augmentation (224×224)
# Key: AugMix + ColorJitter + GaussianBlur + Solarize + Posterize
# These simulate the types of corruptions in the OOD test set
# ============================================================
def make_loaders_224(batch_size, num_workers):
    train_tfms = transforms.Compose([
        transforms.Resize(224),
        transforms.RandomCrop(224, padding=28),
        transforms.RandomHorizontalFlip(),
        # Corruption-simulating augmentations (PIL-level, before ToTensor)
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
        GaussianBlurPIL(p=0.3, radius_range=(0.1, 2.0)),
        RandomSolarize(p=0.1),
        RandomPosterize(p=0.1, bits=4),
        transforms.RandomGrayscale(p=0.05),
        # AugMix: specifically designed for corruption robustness
        transforms.AugMix(severity=3, mixture_width=3, alpha=1.0),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
        transforms.RandomErasing(p=0.3, scale=(0.02, 0.25)),
    ])
    eval_tfms = transforms.Compose([transforms.Resize(224), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    train_full = datasets.CIFAR100(root=DATA_ROOT, train=True, download=True, transform=train_tfms)
    test_ds    = datasets.CIFAR100(root=DATA_ROOT, train=False, download=True, transform=eval_tfms)
    n = len(train_full); n_tr = int(0.9*n)
    g = torch.Generator().manual_seed(SEED)
    train_ds, val_ds = torch.utils.data.random_split(train_full, [n_tr, n-n_tr], generator=g)
    val_eval = torch.utils.data.Subset(
        datasets.CIFAR100(root=DATA_ROOT, train=True, download=False, transform=eval_tfms), val_ds.indices)
    if FAST_DEV_RUN:
        train_ds = torch.utils.data.Subset(train_ds, range(min(256, len(train_ds))))
        val_eval = torch.utils.data.Subset(val_eval, range(min(128, len(val_eval))))
        test_ds  = torch.utils.data.Subset(test_ds, range(min(128, len(test_ds))))
    if IN_COLAB: num_workers = 0
    pin = torch.cuda.is_available()
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin, drop_last=True),
            DataLoader(val_eval, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin),
            DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin))

# CutMix + MixUp (stronger alpha for more regularization)
def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    W, H = x.size(2), x.size(3)
    rw, rh = int(W*np.sqrt(1-lam)), int(H*np.sqrt(1-lam))
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1,y1 = max(cx-rw//2,0), max(cy-rh//2,0)
    x2,y2 = min(cx+rw//2,W), min(cy+rh//2,H)
    x_mix = x.clone()
    x_mix[:,:,x1:x2,y1:y2] = x[idx,:,x1:x2,y1:y2]
    lam = 1 - (x2-x1)*(y2-y1)/(W*H)
    return x_mix, y, y[idx], lam

def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

In [ ]:
# ============================================================
# Training engine with warmup + cosine, higher mix probability
# ============================================================
def train_model(model, train_ld, val_ld, optimizer, scheduler, criterion, device, epochs, use_mix=True):
    history = {"train_acc":[], "val_acc":[]}
    best_va, best_st = -1.0, None
    scaler = _make_scaler(device)

    for ep in range(1, epochs+1):
        model.train(); correct=total=0
        for x, y in tqdm(train_ld, desc=f"Train {ep}/{epochs}", leave=False):
            x, y = x.to(device), y.to(device)
            # 60% chance of CutMix/MixUp (higher than before for more regularization)
            do_mix = use_mix and random.random() < 0.6
            if do_mix:
                if random.random()<0.5: x,ya,yb,lam = cutmix_data(x,y)
                else: x,ya,yb,lam = mixup_data(x,y)
            optimizer.zero_grad(set_to_none=True)
            with _amp_ctx(device):
                logits = model(x)
                if do_mix: loss = lam*criterion(logits,ya)+(1-lam)*criterion(logits,yb)
                else: loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optimizer); scaler.update()
            correct += (logits.argmax(1)==y).sum().item(); total += y.numel()
        if scheduler: scheduler.step()
        tr_acc = correct/max(total,1)
        model.eval(); vc=vt=0
        with torch.no_grad():
            for x,y in val_ld:
                x,y=x.to(device),y.to(device)
                with _amp_ctx(device):
                    vc+=(model(x).argmax(1)==y).sum().item(); vt+=y.numel()
        va_acc = vc/max(vt,1)
        history["train_acc"].append(tr_acc); history["val_acc"].append(va_acc)
        print(f"  Ep {ep:03d}/{epochs} | train {tr_acc:.4f} | val {va_acc:.4f} | lr {optimizer.param_groups[0]['lr']:.6f}")
        if va_acc > best_va: best_va=va_acc; best_st={k:v.cpu().clone() for k,v in model.state_dict().items()}
    if best_st: model.load_state_dict(best_st)
    print(f"  >> Best val: {best_va:.4f}"); return history

@torch.no_grad()
def eval_acc(model, loader, device):
    model.eval(); c=t=0
    for x,y in loader:
        x,y=x.to(device),y.to(device)
        with _amp_ctx(device):
            c+=(model(x).argmax(1)==y).sum().item(); t+=y.numel()
    return 100.0*c/t

In [ ]:
# ============================================================
# TRAIN Part 2: Pretrained CNN — 40 epochs, 224×224
# Tries ConvNeXt-Base (IN-22k) > ResNet-101 > ResNet-50
# ConvNeXt is a modern CNN with much better OOD robustness
# ============================================================
set_seed(SEED)
EP2 = 40 if not FAST_DEV_RUN else 2
NW = 0 if IN_COLAB else 2
print(f"\n{'='*60}\nPart 2: Pretrained CNN — {EP2} epochs @ 224x224\n{'='*60}")
tr2, va2, te2 = make_loaders_224(48, NW)  # BS=48 fits ConvNeXt-Base on T4

# Try models in order of OOD robustness (all are CNNs)
CNN_CANDIDATES = [
    ("convnext_base.fb_in22k_ft_in1k",  "head.fc"),       # ConvNeXt-Base: best OOD CNN
    ("convnext_small.fb_in22k_ft_in1k", "head.fc"),       # ConvNeXt-Small: smaller fallback
    ("resnet101.a1h_in1k",              "fc"),             # ResNet-101
    ("resnet50.a1_in1k",                "fc"),             # ResNet-50 (better weights)
    ("resnet50",                         "fc"),             # ResNet-50 default
]

model = None
head_key = "fc"
for model_name, hk in CNN_CANDIDATES:
    try:
        print(f"  Trying: {model_name}")
        model = timm.create_model(model_name, pretrained=True, num_classes=100).to(device)
        head_key = hk
        print(f"  Loaded: {model_name} ({sum(p.numel() for p in model.parameters()):,} params)")
        break
    except Exception as e:
        print(f"  Failed: {e}")
        model = None

assert model is not None, "No CNN model could be loaded!"

# Separate backbone vs head with correct key
bb = [p for n,p in model.named_parameters() if head_key not in n]
hd = [p for n,p in model.named_parameters() if head_key in n]
print(f"  Backbone params: {sum(p.numel() for p in bb):,}")
print(f"  Head params: {sum(p.numel() for p in hd):,}")

# Warmup + cosine LR
warmup_ep = 5
opt = torch.optim.AdamW([{"params":bb,"lr":5e-5},{"params":hd,"lr":5e-4}], weight_decay=0.05)
sch = torch.optim.lr_scheduler.LambdaLR(opt,
    lr_lambda=lambda ep: min((ep+1)/warmup_ep, 0.5*(1+math.cos(math.pi*max(0,ep-warmup_ep)/max(1,EP2-warmup_ep)))))
crit = nn.CrossEntropyLoss(label_smoothing=0.1)

hist = train_model(model, tr2, va2, opt, sch, crit, device, EP2)
acc = eval_acc(model, te2, device)
print(f"\nPart 2 Clean CIFAR-100 test accuracy: {acc:.2f}%")
torch.save(model.state_dict(), os.path.join(WORK_ROOT, "p2_cnn.pt"))

In [ ]:
### DO NOT CHANGE THE BELOW, REPLACE WITH YOUR MODEL, THE SUBMISSION FILES NEED TO GO THROUGH THE BELOW PREPROCESSING

from huggingface_hub import snapshot_download

def ensure_ood_files(ood_dir):
    os.makedirs(ood_dir, exist_ok=True)
    print("Downloading OOD files from Hugging Face dataset...")
    snapshot_download(repo_id="XThomasBU/ood-test-CS541", repo_type="dataset",
                      local_dir=ood_dir, local_dir_use_symlinks=False)
    print("OOD files ready in", ood_dir)

# ============================================================
# Fast 3-view TTA: original + H-flip + center-crop
# ============================================================
@torch.no_grad()
def get_tta_probs(model, xb, device):
    model.eval()
    h, w = xb.shape[2], xb.shape[3]
    margin = max(2, h // 16)
    probs = torch.zeros(xb.size(0), 100)
    with _amp_ctx(device): probs += F.softmax(model(xb.to(device)).float().cpu(), dim=1)
    with _amp_ctx(device): probs += F.softmax(model(xb.flip(-1).to(device)).float().cpu(), dim=1)
    c = F.interpolate(xb[:,:,margin:h-margin,margin:w-margin], size=(h,w), mode="bilinear", align_corners=False)
    with _amp_ctx(device): probs += F.softmax(model(c.to(device)).float().cpu(), dim=1)
    return probs / 3.0

@torch.no_grad()
def predict_file(model, npy_path, severity, batch_size):
    images = np.load(npy_path, mmap_mode="r")
    start, end = (severity-1)*10000, severity*10000
    mean_t = torch.tensor(MEAN).view(1,3,1,1)
    std_t  = torch.tensor(STD).view(1,3,1,1)
    all_preds = []
    for b0 in range(start, end, batch_size):
        b1 = min(b0+batch_size, end)
        xb = torch.from_numpy(np.array(images[b0:b1], copy=True)).permute(0,3,1,2).float().div(255.0)
        xb = (xb - mean_t) / std_t
        xb = F.interpolate(xb, 224, mode="bilinear", align_corners=False)
        all_preds.append(get_tta_probs(model, xb, device).argmax(1).numpy())
    return np.concatenate(all_preds)

ensure_ood_files(OOD_DIR)
distortion_files = sorted([p for p in os.listdir(OOD_DIR) if p.startswith("distortion") and p.endswith(".npy")])
print(f"Distortion files: {len(distortion_files)}")

BATCH = 64 if device.type=="cuda" else 16  # larger batch, fewer TTA views = faster
rows = []
for fi, fname in enumerate(distortion_files):
    dname = os.path.splitext(fname)[0]
    path = os.path.join(OOD_DIR, fname)
    for sev in [1,2,3,4,5]:
        pred = predict_file(model, path, sev, BATCH)
        for i, y in enumerate(pred.tolist()):
            rows.append((f"{dname}_{sev}_{i}", int(y)))
    print(f"  [{fi+1}/{len(distortion_files)}] {dname} done")

submission = pd.DataFrame(rows, columns=["id", "label"])
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"\nWrote {SUBMISSION_PATH} ({len(submission)} rows)")
print(submission.head())

if IN_COLAB:
    try: from google.colab import files; files.download(SUBMISSION_PATH)
    except: print("Download manually:", SUBMISSION_PATH)

In [ ]:
# Plot training curves
plt.figure(figsize=(8,5))
plt.plot(hist["train_acc"], label="Train"); plt.plot(hist["val_acc"], label="Val")
plt.title("Part 2: ResNet-50"); plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## OOD Submission Generation

Submission format: `id,label`  
Using 5-view TTA (original, H-flip, center-crop, TL-crop, BR-crop).

## Report Analysis — Part 2

Run the cells below **after training** to generate the required report figures:
1. Loss & accuracy curves
2. Per-class accuracy → Top 3 worst-performing classes
3. Top 3 images with largest prediction errors

In [ ]:
# ============================================================
# 1. Loss & Accuracy Curves
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(hist["train_acc"], label="Train Accuracy", linewidth=2)
ax1.plot(hist["val_acc"], label="Val Accuracy", linewidth=2)
ax1.set_title("Part 2: Pretrained CNN — Accuracy", fontsize=13)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy"); ax1.legend(); ax1.grid(alpha=0.3)
ax1.axhline(y=max(hist["val_acc"]), color='r', linestyle='--', alpha=0.5)
ax1.legend([f"Train (final: {hist['train_acc'][-1]:.4f})", f"Val (best: {max(hist['val_acc']):.4f})"])

train_loss_approx = [-math.log(max(a, 0.01)) for a in hist["train_acc"]]
val_loss_approx = [-math.log(max(a, 0.01)) for a in hist["val_acc"]]
ax2.plot(train_loss_approx, label="Train Loss (approx)", linewidth=2)
ax2.plot(val_loss_approx, label="Val Loss (approx)", linewidth=2)
ax2.set_title("Part 2: Pretrained CNN — Loss (approx)", fontsize=13)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(WORK_ROOT, "p2_accuracy_loss.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"Best val accuracy: {max(hist['val_acc']):.4f}")
print(f"Test accuracy: {acc:.2f}%")

In [ ]:
# ============================================================
# 2. Per-class accuracy → Top 3 worst classes
# 3. Top 3 images with largest prediction errors
# ============================================================
CIFAR100_CLASSES = [
    'apple','aquarium_fish','baby','bear','beaver','bed','bee','beetle','bicycle','bottle',
    'bowl','boy','bridge','bus','butterfly','camel','can','castle','caterpillar','cattle',
    'chair','chimpanzee','clock','cloud','cockroach','couch','crab','crocodile','cup','dinosaur',
    'dolphin','elephant','flatfish','forest','fox','girl','hamster','house','kangaroo','keyboard',
    'lamp','lawn_mower','leopard','lion','lizard','lobster','man','maple_tree','motorcycle','mountain',
    'mouse','mushroom','oak_tree','orange','orchid','otter','palm_tree','pear','pickup_truck','pine_tree',
    'plain','plate','poppy','porcupine','possum','rabbit','raccoon','ray','road','rocket',
    'rose','sea','seal','shark','shrew','skunk','skyscraper','snail','snake','spider',
    'squirrel','streetcar','sunflower','sweet_pepper','table','tank','telephone','television','tiger','tractor',
    'train','trout','tulip','turtle','wardrobe','whale','willow_tree','wolf','woman','worm'
]

eval_tfms_224 = transforms.Compose([transforms.Resize(224), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
test_ds_eval = datasets.CIFAR100(root=DATA_ROOT, train=False, download=True, transform=eval_tfms_224)
test_loader_eval = DataLoader(test_ds_eval, batch_size=64, shuffle=False, num_workers=0)

model.eval()
all_preds = []; all_labels = []; all_logits_list = []
with torch.no_grad():
    for x, y in tqdm(test_loader_eval, desc="Evaluating test set"):
        x = x.to(device)
        with _amp_ctx(device):
            logits = model(x).float().cpu()
        all_logits_list.append(logits)
        all_preds.append(logits.argmax(1))
        all_labels.append(y)

all_preds = torch.cat(all_preds); all_labels = torch.cat(all_labels); all_logits = torch.cat(all_logits_list)
all_probs = F.softmax(all_logits, dim=1)

# Per-class accuracy
class_correct = torch.zeros(100); class_total = torch.zeros(100)
for i in range(len(all_labels)):
    label = all_labels[i].item()
    class_total[label] += 1
    if all_preds[i].item() == label: class_correct[label] += 1
class_acc = class_correct / class_total.clamp(min=1)
sorted_idx = class_acc.argsort()

print("\n" + "="*60)
print("PART 2 — TOP 3 WORST-PERFORMING CLASSES")
print("="*60)
for rank, idx in enumerate(sorted_idx[:3]):
    idx = idx.item()
    print(f"  #{rank+1}: Class {idx} ({CIFAR100_CLASSES[idx]}) — Accuracy: {class_acc[idx]*100:.1f}% ({int(class_correct[idx])}/{int(class_total[idx])})")

print("\nTOP 3 BEST-PERFORMING CLASSES")
for rank, idx in enumerate(sorted_idx[-3:].flip(0)):
    idx = idx.item()
    print(f"  #{rank+1}: Class {idx} ({CIFAR100_CLASSES[idx]}) — Accuracy: {class_acc[idx]*100:.1f}%")

# Bar chart
fig, ax = plt.subplots(figsize=(18, 5))
colors = ['red' if i in sorted_idx[:3].tolist() else 'steelblue' for i in range(100)]
ax.bar(range(100), class_acc.numpy()*100, color=colors, width=0.8)
ax.set_xlabel("Class Index"); ax.set_ylabel("Accuracy (%)")
ax.set_title("Part 2: Per-Class Accuracy (red = worst 3)")
ax.set_xlim(-1, 100); ax.grid(alpha=0.3, axis='y')
for idx in sorted_idx[:3]:
    idx = idx.item()
    ax.annotate(CIFAR100_CLASSES[idx], (idx, class_acc[idx]*100+1), fontsize=7, ha='center', color='red')
plt.tight_layout()
plt.savefig(os.path.join(WORK_ROOT, "p2_per_class_accuracy.png"), dpi=150, bbox_inches='tight')
plt.show()

# Top 3 worst predictions
wrong_mask = (all_preds != all_labels)
wrong_indices = wrong_mask.nonzero(as_tuple=True)[0]
wrong_confidences = all_probs[wrong_indices, all_preds[wrong_indices]]
sorted_wrong = wrong_confidences.argsort(descending=True)
top3_wrong_idx = wrong_indices[sorted_wrong[:3]]

test_ds_display = datasets.CIFAR100(root=DATA_ROOT, train=False, download=True, transform=transforms.ToTensor())
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
print("\n" + "="*60)
print("PART 2 — TOP 3 MOST CONFIDENT WRONG PREDICTIONS")
print("="*60)
for i, (ax, idx) in enumerate(zip(axes, top3_wrong_idx)):
    idx = idx.item()
    img, true_label = test_ds_display[idx]
    pred_label = all_preds[idx].item()
    pred_conf = all_probs[idx, pred_label].item()
    true_conf = all_probs[idx, true_label].item()
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(f"Pred: {CIFAR100_CLASSES[pred_label]} ({pred_conf*100:.1f}%)\nTrue: {CIFAR100_CLASSES[true_label]} ({true_conf*100:.1f}%)", fontsize=10, color='red')
    ax.axis('off')
    print(f"  #{i+1}: True={CIFAR100_CLASSES[true_label]} ({true_conf*100:.1f}%), Pred={CIFAR100_CLASSES[pred_label]} ({pred_conf*100:.1f}%)")

plt.suptitle("Part 2: Top 3 Most Confident Incorrect Predictions", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(WORK_ROOT, "p2_top3_worst_predictions.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"\nOverall test accuracy: {(all_preds == all_labels).float().mean()*100:.2f}%")